
## Notebook 1: Data Collection & Preparation

This notebook is adapted from the *Detection of Forest Loss in Borneo using
K-means Unsupervised Learning* notebook on the GEOL0069 Jupyter Notebook — a
project originally developed by Michael Telespher that demonstrates how to
fetch and analyse satellite data from Google Earth Engine.

Unsupervised (GMM) and supervised (CNN) classification are implemented across
Notebooks 1–3 to detect and quantify water body loss in the Aral Sea basin.
Sentinel-2 imagery is used for its high spatial resolution and comprehensive
spectral bands, accessed via Google Earth Engine (GEE).


## Set Up
Run this cell first. The timer starts here and is used to calculate the environmental cost at the end of the notebook.

In [ ]:
import time
notebook_start = time.time()

Install required packages and authenticate with Google Earth Engine before running this notebook.

1. Sign up at https://earthengine.google.com/
2. Create a Google Cloud project at https://console.cloud.google.com
3. Enable the Earth Engine API for your project
4. Register your project with Earth Engine
5. Replace 'your-project-id' in the cell below with your actual project ID

In [2]:
# Install required packages
# pip install earthengine-api
# pip install geemap
# pip install pandas numpy matplotlib scikit-learn rasterio

import time
start_time = time.time()  # for environmental cost tracking

import ee
import numpy as np
import geemap
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [3]:

ee.Authenticate()
ee.Initialize(project='aral-sea-496719')

## Define Study Region

The bounding box encompasses the South Aral Sea and the remaining eastern basin, where the most severe shrinkage has taken place.


In [4]:
study_region = {
    'aral_sea': {
        'geometry': ee.Geometry.Rectangle([57.5, 43.0, 61.5, 46.5]),
        'country': 'Kazakhstan / Uzbekistan',
        'description': 'Aral Sea basin — water body collapse'
    }
}

# Quick visual check of the study region
from IPython.display import Image, display

# Define visualization parameters for the geometry (e.g., outline color)
geometry_vis_params = {'color': 'FF0000'} # Red color for the outline

# Create an image to paint the geometry onto.
# We'll paint the boundary of the geometry with value 1 and a line width of 2.
painted_geometry_image = ee.Image().paint(study_region['aral_sea']['geometry'], 1, 2)

# Get a thumbnail URL for this painted image, using the geometry's bounds as the region.
# The 'min' and 'max' are for the visualization of the painted image (0 for no geometry, 1 for geometry).
# The 'palette' maps these values to colors (transparent for background, red for geometry).
url = painted_geometry_image.getThumbURL({
    'region': study_region['aral_sea']['geometry'].bounds(),
    'dimensions': 512,
    'format': 'png',
    'min': 0,
    'max': 1,
    'palette': ['00000000', geometry_vis_params['color']] # Transparent for background, red for geometry
})

print(f"Study region: {study_region['aral_sea']['description']}")
print(f"Country: {study_region['aral_sea']['country']}")
print(f"Bounds: {study_region['aral_sea']['geometry'].bounds().getInfo()}")



Study region: Aral Sea basin — water body collapse
Country: Kazakhstan / Uzbekistan
Bounds: {'geodesic': False, 'type': 'Polygon', 'coordinates': [[[57.5, 42.99999999999997], [61.50000000000001, 42.99999999999997], [61.50000000000001, 46.517432635014245], [57.5, 46.517432635014245], [57.5, 42.99999999999997]]]}


## Image Preprocessing

### Cloud Masking

Cloud masking removes pixels contaminated by clouds and shadows before analysis.
Sentinel-2 provides a Scene Classification Layer (SCL) which labels every pixel,
allowing us to identify and exclude cloudy pixels so that only clear, usable
imagery is retained.



In [5]:

def mask_sentinel2_clouds(image):
    """Simplified cloud masking using the Scene Classification Layer (SCL).
    Excludes cloud shadows (3), medium probability cloud (8),
    high probability cloud (9), and cirrus (10).
    """
    scl = image.select('SCL')
    clear_mask = scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10))
    return image.updateMask(clear_mask)

### Spectral Indices

For the Aral Sea, water-focused indices are needed:

- **NDWI** = detects open water, positive values indicate water (McFeeters, 1996)
- **MNDWI** = separates water from salt flat and bare soil (Xu, 2006)
- **NDVI** = distinguishes desert scrub from bare salt crust

In [6]:
def add_indices(image):
    """Add NDWI, MNDWI and NDVI to a Sentinel-2 image.

    Sentinel-2 bands:
      B3  = Green, B4 = Red, B8 = NIR, B11 = SWIR1
    """
    # NDWI — primary open water detector
    ndwi = image.normalizedDifference(['B3', 'B8']).rename('NDWI')

    # MNDWI — separates water from salt flat in arid environments
    mndwi = image.normalizedDifference(['B3', 'B11']).rename('MNDWI')

    # NDVI
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')

    return image.addBands([ndwi, mndwi, ndvi])

## Image Collection

This cell fetches Sentinel-2 imagery for four time periods (2015, 2018, 2021, 2024)
and builds a single cloud-free composite image for each year. A July to September
window is used to avoid dust storms and snow over Central Asia. A median composite
is taken across all valid scenes in each window, which further reduces any remaining
cloud artefacts.

In [7]:
years = [2015, 2018, 2021, 2024]

region = study_region['aral_sea']['geometry']
composites = {}

for year in years:
    print(f'Collecting imagery for {year}...')

    start = f'{year}-07-01'
    end   = f'{year}-09-30'


    collection = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
        .filterDate(start, end) \
        .filterBounds(region) \
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 50)) \
        .map(mask_sentinel2_clouds) \
        .map(add_indices)

    count = collection.size().getInfo()
    print(f'  Found {count} images')


    processed = collection \
        .select(['B2', 'B3', 'B4', 'B8', 'B11', 'B12', 'NDWI', 'MNDWI', 'NDVI']) \
        .median() \
        .multiply(ee.Image([0.0001, 0.0001, 0.0001, 0.0001, 0.0001, 0.0001, 1, 1, 1])) \
        .clip(region)

    composites[year] = processed
    print(f'  {year}: composite ready')

print(f'\nSuccessfully collected composites for {len(composites)} years!')

  Found 13 images
  2015: composite ready
  Found 75 images
  2018: composite ready
  Found 570 images
  2021: composite ready
  Found 608 images
  2024: composite ready

Successfully collected composites for 4 years!


## Visualise the Study Area



In [8]:
from IPython.display import Image, display

rgb_vis = {
    'bands': ['B4', 'B3', 'B2'],
    'min': 0,
    'max': 0.3
}

print('ARAL SEA SENTINEL-2 IMAGERY')
print('=' * 40)

for year in years:
    print(f'\n{year}:')

    url = composites[year].visualize(**rgb_vis).getThumbURL({
        'region': region,
        'dimensions': 512,
        'format': 'png'
    })

    display(Image(url=url, width=400))

ARAL SEA SENTINEL-2 IMAGERY

2015:



2018:



2021:



2024:


## Export Index Stacks to Google Drive

This cell exports the NDWI, MNDWI and NDVI layers for each year to Google Drive
as GeoTIFF files — a standard format for storing satellite imagery with geographic
information. These files are loaded in Notebooks 2 and 3 for classification.

Check the GEE Tasks tab to confirm exports have completed before moving to Notebook 2

In [ ]:
# Export index stacks for each year
print('Starting GEE export tasks...')

for year in years:
    export_image = composites[year].select(['NDWI', 'MNDWI', 'NDVI'])

    task = ee.batch.Export.image.toDrive(
        image=export_image,
        description=f'aral_sea_indices_{year}',
        folder='aral_sea',
        fileNamePrefix=f'aral_sea_indices_{year}',
        region=region,
        scale=500,
        crs='EPSG:4326',
        maxPixels=1e9
    )
    task.start()
    print(f'  Export started: aral_sea_indices_{year}.tif')

print('\nAll export tasks started — check the GEE Tasks tab to monitor progress')

Starting GEE export tasks...
  Export started: aral_sea_indices_2015.tif
  Export started: aral_sea_indices_2018.tif
  Export started: aral_sea_indices_2021.tif
  Export started: aral_sea_indices_2024.tif

All export tasks started — check the GEE Tasks tab to monitor progress


## Environmental Cost

Computational energy usage and estimated carbon emissions are tracked for each notebook.
Runtime is recorded manually and used to estimate energy consumption, assuming a 20W CPU,
UK grid carbon intensity of 0.233 kg CO2/kWh, and an electricity cost of £0.30/kWh.
Update the runtime value below after running the notebook.

In [ ]:
# Assumptions: 20W CPU, UK grid intensity 0.233 kg CO2/kWh, £0.30/kWh

runtime_mins = (time.time() - notebook_start) / 60
energy_kwh   = 0.020 * (runtime_mins / 60)
co2_kg       = energy_kwh * 0.233
cost_gbp     = energy_kwh * 0.30

print('=== Notebook 1 Environmental Cost ===')
print(f'Runtime:  {runtime_mins} min')
print(f'Energy:   {energy_kwh * 1000:.4f} Wh')
print(f'CO2e:     {co2_kg * 1000:.4f} g')
print(f'Cost:     £{cost_gbp:.4f}')

=== Notebook 1 Environmental Cost ===
Runtime:  0.6122053345044454 min
Energy:   0.2041 Wh
CO2e:     0.0475 g
Cost:     £0.0001
